### Project Objective

This project uses SQL and Tableau to analyse supply chain service performance and commercial exposure. The analysis identifies delivery-performance gaps, evaluates their commercial significance, and translates the findings into management priorities.

In [46]:

import pandas as pd

file_path = "/Users/shivanimaithani/Downloads/DataCoSupplyChainDataset_UTF8.csv"

df = pd.read_csv(
    file_path,
    encoding="utf-8"
)

df.head()

,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


In [48]:
df.shape

(180519, 53)

## Dataset Overview

The DataCo SMART Supply Chain dataset contains 180,519 transaction-level records across 53 columns.

The dataset includes information on orders, customers, products, delivery performance, sales, shipping, and geographic markets.

### Data Grain

The analysis is conducted at **order-item / transaction level**. A single Order ID can contain multiple Order Item IDs.

### Key Fields Used

- Order Id
- Order Status
- Delivery Status
- DateOrders
- Sales
- Order Profit Per Order
- Shipping Mode
- Market
- Order Region
- Category Name
- Customer Segment
- Days for shipping (real)
- Days for shipment (scheduled)

In [50]:
# Basic dataset overview

print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

print("\nColumn names:")
print(df.columns.tolist())

print("\nDataset information:")
df.info()

Number of rows: 180519
Number of columns: 53

Column names:
['Type', 'Days for shipping (real)', 'Days for shipment (scheduled)', 'Benefit per order', 'Sales per customer', 'Delivery Status', 'Late_delivery_risk', 'Category Id', 'Category Name', 'Customer City', 'Customer Country', 'Customer Email', 'Customer Fname', 'Customer Id', 'Customer Lname', 'Customer Password', 'Customer Segment', 'Customer State', 'Customer Street', 'Customer Zipcode', 'Department Id', 'Department Name', 'Latitude', 'Longitude', 'Market', 'Order City', 'Order Country', 'Order Customer Id', 'order date (DateOrders)', 'Order Id', 'Order Item Cardprod Id', 'Order Item Discount', 'Order Item Discount Rate', 'Order Item Id', 'Order Item Product Price', 'Order Item Profit Ratio', 'Order Item Quantity', 'Sales', 'Order Item Total', 'Order Profit Per Order', 'Order Region', 'Order State', 'Order Status', 'Order Zipcode', 'Product Card Id', 'Product Category Id', 'Product Description', 'Product Image', 'Product Name',

## Data Audit

Before performing the analysis, the dataset was audited to assess data completeness, record uniqueness, data structure, date integrity, and key business fields relevant to delivery and commercial performance.

### Missing Values

- The dataset contains **180,519 order-item records across 53 columns**.
- `Product Description` is completely missing (100%) and is not required for the analysis.
- `Order Zipcode` has **86.24% missing values** and will not be used for geographic analysis.
- Geographic analysis will instead use available fields such as `Market`, `Order Region`, `Order State`, and `Order City`.

### Record Structure

- There are **no complete duplicate rows**.
- Each `Order Item Id` is unique across the dataset.
- The dataset contains **65,752 unique orders** and **20,652 unique customers**.
- One order can contain multiple order items, confirming that the dataset is at **order-item / transaction level**.

### Date Integrity

- Order dates range from **January 2015 to January 2018**.
- Shipping dates occur after order dates, with no records showing shipping before the order date.
- Date fields are therefore suitable for analysing delivery performance over time.

### Delivery Status

`Delivery Status` contains four categories:

- Late delivery
- Advance shipping
- Shipping on time
- Shipping canceled

Late delivery is the largest category, making delivery reliability the primary operational performance issue investigated in this project.

In [53]:
#  Missing Value Audit

missing = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_%": (df.isnull().sum() / len(df) * 100).round(2)
})

missing = missing[missing["Missing_Count"] > 0] \
    .sort_values("Missing_Count", ascending=False)

missing

,Missing_Count,Missing_%
Product Description,180519,100.00
Order Zipcode,155679,86.24
Customer Lname,8,0.00
Customer Zipcode,3,0.00


In [55]:
#  Duplicate Value Audit
print("Duplicate complete rows:", df.duplicated().sum())

print("\nOrder Item ID")
print("Total rows:", len(df))
print("Unique Order Item IDs:", df["Order Item Id"].nunique())

print("\nOrder ID")
print("Unique Order IDs:", df["Order Id"].nunique())

print("\nCustomer ID")
print("Unique Customer IDs:", df["Customer Id"].nunique())

print("\nProduct Card ID")
print("Unique Product Card IDs:", df["Product Card Id"].nunique())

Duplicate complete rows: 0

Order Item ID
Total rows: 180519
Unique Order Item IDs: 180519

Order ID
Unique Order IDs: 65752

Customer ID
Unique Customer IDs: 20652

Product Card ID
Unique Product Card IDs: 118


In [57]:
#Average items per order
print("Average items per order:",
      round(len(df) / df["Order Id"].nunique(), 2))

print("\nOrders with multiple items:")

order_items = df.groupby("Order Id")["Order Item Id"].count()

print("Orders with 1 item:", (order_items == 1).sum())
print("Orders with >1 item:", (order_items > 1).sum())

print("\nMaximum items in a single order:",
      order_items.max())

Average items per order: 2.75

Orders with multiple items:
Orders with 1 item: 19850
Orders with >1 item: 45902

Maximum items in a single order: 5


In [59]:
# Date Quality Audit

# Convert date columns
df["order_date"] = pd.to_datetime(
    df["order date (DateOrders)"],
    errors="coerce"
)

df["shipping_date"] = pd.to_datetime(
    df["shipping date (DateOrders)"],
    errors="coerce"
)

# Date audit
print("ORDER DATE")
print("Earliest:", df["order_date"].min())
print("Latest:", df["order_date"].max())
print("Missing:", df["order_date"].isna().sum())

print("\nSHIPPING DATE")
print("Earliest:", df["shipping_date"].min())
print("Latest:", df["shipping_date"].max())
print("Missing:", df["shipping_date"].isna().sum())

# Check impossible date relationships
print("\nShipping before order:")
print((df["shipping_date"] < df["order_date"]).sum())

ORDER DATE
Earliest: 2015-01-01 00:00:00
Latest: 2018-01-31 23:38:00
Missing: 0

SHIPPING DATE
Earliest: 2015-01-03 00:00:00
Latest: 2018-02-06 22:14:00
Missing: 0

Shipping before order:
0


In [61]:
# Categorical Value Audit
categorical_columns = [
    "Type",
    "Delivery Status",
    "Customer Segment",
    "Market",
    "Order Region",
    "Order Status",
    "Shipping Mode",
    "Department Name",
    "Category Name"
]

for col in categorical_columns:
    print("\n" + "="*60)
    print(col)
    print("="*60)
    print(df[col].value_counts(dropna=False))


Type
Type
DEBIT       69295
TRANSFER    49883
PAYMENT     41725
CASH        19616
Name: count, dtype: int64

Delivery Status
Delivery Status
Late delivery        98977
Advance shipping     41592
Shipping on time     32196
Shipping canceled     7754
Name: count, dtype: int64

Customer Segment
Customer Segment
Consumer       93504
Corporate      54789
Home Office    32226
Name: count, dtype: int64

Market
Market
LATAM           51594
Europe          50252
Pacific Asia    41260
USCA            25799
Africa          11614
Name: count, dtype: int64

Order Region
Order Region
Central America    28341
Western Europe     27109
South America      14935
Oceania            10148
Northern Europe     9792
Southeast Asia      9539
Southern Europe     9431
Caribbean           8318
West of USA         7993
South Asia          7731
Eastern Asia        7280
East of USA         6915
West Asia           6009
US Center           5887
South of  USA       4045
Eastern Europe      3920
West Africa         36

In [63]:
# Numerical Quality Audit

numeric_columns = [

    "Days for shipping (real)",

    "Days for shipment (scheduled)",

    "Benefit per order",

    "Sales per customer",

    "Late_delivery_risk",

    "Order Item Discount",

    "Order Item Discount Rate",

    "Order Item Product Price",

    "Order Item Profit Ratio",

    "Order Item Quantity",

    "Sales",

    "Order Item Total",

    "Order Profit Per Order",

    "Product Price"

]

numeric_audit = pd.DataFrame({

    "Min": df[numeric_columns].min(),

    "Max": df[numeric_columns].max(),

    "Mean": df[numeric_columns].mean().round(2),

    "Missing": df[numeric_columns].isna().sum()

})

numeric_audit

,Min,Max,Mean,Missing
Days for shipping (real),0.00000,6.000000,3.50,0
Days for shipment (scheduled),0.00000,4.000000,2.93,0
Benefit per order,-4274.97998,911.799988,21.97,0
Sales per customer,7.49000,1939.989990,183.11,0
Late_delivery_risk,0.00000,1.000000,0.55,0
Order Item Discount,0.00000,500.000000,20.66,0
Order Item Discount Rate,0.00000,0.250000,0.10,0
Order Item Product Price,9.99000,1999.989990,141.23,0
Order Item Profit Ratio,-2.75000,0.500000,0.12,0
Order Item Quantity,1.00000,5.000000,2.13,0


In [65]:
# Check relationships between important financial fields

df[
    [
        "Sales",
        "Order Item Total",
        "Order Profit Per Order",
        "Benefit per order",
        "Order Item Discount",
        "Order Item Discount Rate",
        "Order Item Product Price",
        "Order Item Quantity"
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
Sales,180519.0,203.772096,132.273077,9.99000,119.980003,199.919998,299.950012,1999.989990
Order Item Total,180519.0,183.107609,120.043670,7.49000,104.379997,163.990005,247.399994,1939.989990
Order Profit Per Order,180519.0,21.974989,104.433526,-4274.97998,7.000000,31.520000,64.800003,911.799988
Benefit per order,180519.0,21.974989,104.433526,-4274.97998,7.000000,31.520000,64.800003,911.799988
Order Item Discount,180519.0,20.664741,21.800901,0.00000,5.400000,14.000000,29.990000,500.000000
Order Item Discount Rate,180519.0,0.101668,0.070415,0.00000,0.040000,0.100000,0.160000,0.250000
Order Item Product Price,180519.0,141.232550,139.732492,9.99000,50.000000,59.990002,199.990005,1999.989990
Order Item Quantity,180519.0,2.127638,1.453451,1.00000,1.000000,1.000000,3.000000,5.000000


In [67]:
# Check whether Benefit per order and Order Profit Per Order are identical

(df["Benefit per order"] == df["Order Profit Per Order"]).mean()

1.0

In [69]:
# Check whether Sales and Order Item Total are identical

(df["Sales"] == df["Order Item Total"]).mean()

0.055550939236313074

In [75]:
# Check for leading/trailing whitespace

categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    whitespace_count = (
        df[col].astype(str).str.strip() != df[col].astype(str)
    ).sum()
    
    if whitespace_count > 0:
        print(f"{col}: {whitespace_count} values with whitespace")

Category Name: 1475 values with whitespace
Customer Street: 1829 values with whitespace
Department Name: 362 values with whitespace
Order Region: 17925 values with whitespace
Product Image: 886 values with whitespace
Product Name: 1774 values with whitespace


## Data Cleaning & Preparation

In [77]:
# Clean whitespace from all text columns

categorical_columns = df.select_dtypes(include="object").columns

for col in categorical_columns:
    df[col] = df[col].str.strip()

print("Whitespace cleaning completed.")

Whitespace cleaning completed.


In [79]:
for col in categorical_columns:
    whitespace_count = (
        df[col].astype(str).str.strip() != df[col].astype(str)
    ).sum()
    
    if whitespace_count > 0:
        print(f"{col}: {whitespace_count}")

In [101]:
# Order Status Analysis

status_analysis = (

    df.groupby("Order Status")

    .agg(

        Orders=("Order Id", "nunique"),

        Order_Items=("Order Item Id", "count"),

        Sales=("Sales", "sum"),

        Profit=("Order Profit Per Order", "sum")

    )

    .sort_values("Sales", ascending=False)

)

status_analysis["Sales_%"] = (

    status_analysis["Sales"]

    / status_analysis["Sales"].sum()

    * 100

).round(2)

status_analysis

,Orders,Order_Items,Sales,Profit,Sales_%
Order Status,,,,,
COMPLETE,21716,59491,1.209531e+07,1.321736e+06,32.88
PENDING_PAYMENT,14382,39832,8.106698e+06,8.438102e+05,22.04
PROCESSING,7901,21902,4.504064e+06,4.948259e+05,12.24
PENDING,7321,20227,4.120533e+06,4.357259e+05,11.20
CLOSED,7249,19616,4.022624e+06,4.579811e+05,10.94
ON_HOLD,3624,9804,1.981543e+06,2.089130e+05,5.39
SUSPECTED_FRAUD,1488,4062,8.259350e+05,8.513671e+04,2.25
CANCELED,1367,3692,7.443704e+05,7.534563e+04,2.02
PAYMENT_REVIEW,704,1893,3.836537e+05,4.342879e+04,1.04


In [103]:
df[df["Order Status"].isin(["CANCELED", "SUSPECTED_FRAUD"])][

    ["Order Status", "Order Id", "Sales", "Order Profit Per Order"]

].describe()

,Order Id,Sales,Order Profit Per Order
count,7754.000000,7754.000000,7754.000000
mean,36371.793010,202.515522,20.696717
std,21249.830354,129.585252,106.160006
min,50.000000,11.290000,-1716.000000
25%,17790.250000,119.980003,6.600000
50%,36437.000000,199.919998,31.565001
75%,54869.000000,299.950012,64.975000
max,77184.000000,1500.000000,675.000000


In [ ]:
"""Checking if same order has two status"""

In [105]:
status_per_order = (
    df.groupby("Order Id")["Order Status"]
      .nunique()
)

status_per_order.value_counts().sort_index()

Order Status
1    65752
Name: count, dtype: int64

In [107]:
multi_status_orders = status_per_order[status_per_order > 1]

len(multi_status_orders)

0

In [111]:
#We cannot confidently present SUM(Sales) as “Revenue” without first deciding how canceled, fraudulent, and potentially pending transactions should be treated.

# Step: Define Revenue-Eligible Transactions

# Purpose: Identify transactions that will be included in  realized revenue KPIs

revenue_statuses = ["COMPLETE", "CLOSED"]

df["Revenue_Eligible"] = df["Order Status"].isin(revenue_statuses)

df["Revenue_Eligible"].value_counts()

Revenue_Eligible
False    101412
True      79107
Name: count, dtype: int64

In [119]:
# Revenue Definition & Sales Treatment

# Total sales recorded across all transactions

total_recorded_sales = df["Sales"].sum()

# Realised sales: COMPLETE and CLOSED transactions only

realized_sales = df.loc[

    df["Revenue_Eligible"],

    "Sales"

].sum()

# Sales excluded from the realised-sales definition

excluded_sales = total_recorded_sales - realized_sales

# Percentage of recorded sales excluded

excluded_sales_pct = (

    excluded_sales / total_recorded_sales * 100

)

# Percentage of recorded sales classified as realised

realized_sales_pct = (

    realized_sales / total_recorded_sales * 100

)

print("Total Recorded Sales: $", round(total_recorded_sales, 2))

print("Realised Sales: $", round(realized_sales, 2))

print("Realised Sales %:", round(realized_sales_pct, 2), "%")

print("Excluded Sales: $", round(excluded_sales, 2))

print("Excluded Sales %:", round(excluded_sales_pct, 2), "%")

Total Recorded Sales: $ 36784735.01
Realised Sales: $ 16117939.12
Realised Sales %: 43.82 %
Excluded Sales: $ 20666795.9
Excluded Sales %: 56.18 %


## SQL Environment Setup

DuckDB is used to run SQL queries directly within the Jupyter Notebook.

The cleaned Pandas DataFrame is registered as a DuckDB table, allowing the analysis to be performed using SQL while retaining the interactive Jupyter environment.

This approach is used to calculate the core supply chain and commercial KPIs consistently before visualising the results in Tableau.

In [127]:
# ============================================================

# SQL Environment Setup

# Purpose: Enable SQL queries directly inside Jupyter Notebook

#          using DuckDB.

# ============================================================

!pip install duckdb

In [128]:
# ============================================================

# SQL Environment Setup

# Purpose: Connect DuckDB to the existing Pandas DataFrame

#          so we can run SQL queries directly in Jupyter.

# ============================================================

import duckdb

con = duckdb.connect()

con.register("supply_chain", df)

In [131]:
# ============================================================

# SQL Connection Test

# Purpose: Confirm that DuckDB can query our dataset.

# ============================================================

con.execute("""

    SELECT COUNT(*) AS total_rows

    FROM supply_chain

""").df()

,total_rows
0,180519


## Core KPI Analysis

In [159]:
# KPI: Realized Sales

# Definition: Sales from orders with COMPLETE or CLOSED status.

# Purpose: Exclude non-realized order statuses from the revenue proxy.
con.execute("""
Select 
  Round(SUM(Sales) / 1000000 ,2) AS Realized_Sales

from supply_chain
where "Order Status" = 'CLOSED' 
Or "Order Status" = 'COMPLETE'

""").df()

,Realized_Sales
0,16.12


In [157]:
# KPI: Realized Orders

# Definition: Unique orders with COMPLETE or CLOSED status.
con.execute("""
Select Count(Distinct"Order ID") as  Realized_Orders
from supply_chain
where "Order Status" = 'CLOSED' 
Or "Order Status" = 'COMPLETE'

""").df()



,Realized_Orders
0,28965


In [155]:
# KPI: Realized Profit

# Definition: Total profit from orders with COMPLETE or CLOSED status.

con.execute("""

SELECT 

    Round(SUM("Order Profit Per Order") / 1000000 ,2) AS Realized_Profit

FROM supply_chain

WHERE "Order Status" IN ('CLOSED', 'COMPLETE')

""").df()

,Realized_Profit
0,1.78


In [153]:
# KPI: Realized Profit Margin
# Definition: Realized profit as a percentage of realized sales.
con.execute("""
Select 
 ROUND(SUM("Order Profit Per Order") / 1000000.0, 2) AS Realized_Profit_M,
    ROUND(SUM(Sales) / 1000000.0, 2) AS Realized_Sales_M,
Round(Sum("Order Profit Per Order")/Sum(Sales)*100,2) as  Realized_Profit_Margin_Perc
from supply_chain
where "Order Status" = 'CLOSED' 
Or "Order Status" = 'COMPLETE'

""").df()


,Realized_Profit_M,Realized_Sales_M,Realized_Profit_Margin_Perc
0,1.78,16.12,11.04


In [151]:
# KPI: Late Delivery Rate
# Definition: Percentage of realized orders classified as "Late delivery".
# Purpose: Measure delivery reliability for completed/closed orders.


con.execute("""

Select Round((Count(Distinct"Order ID")/28965)*100,2)
 as  Late_Delivery_perc,
from supply_chain
where( "Order Status" = 'CLOSED' 
Or "Order Status" = 'COMPLETE') 
and "Delivery Status"='Late delivery'
""").df()


,Late_Delivery_perc
0,57.42


## Delivery Performance Analysis

In [161]:
#KPI: Late Delivery Rate by Shipping Mode

#Business question:

#Which shipping modes have the highest late-delivery rate?
con.execute("""

SELECT

    "Shipping Mode",

    COUNT(DISTINCT "Order ID") AS Total_Realized_Orders,

    COUNT(DISTINCT CASE

        WHEN "Delivery Status" = 'Late delivery'

        THEN "Order ID"

    END) AS Late_Realized_Orders,

    ROUND(

        COUNT(DISTINCT CASE

            WHEN "Delivery Status" = 'Late delivery'

            THEN "Order ID"

        END) * 100.0

        / COUNT(DISTINCT "Order ID"),

        2

    ) AS Late_Delivery_Rate

FROM supply_chain

WHERE "Order Status" IN ('CLOSED', 'COMPLETE')

GROUP BY "Shipping Mode"

ORDER BY Late_Delivery_Rate DESC

""").df()


,Shipping Mode,Total_Realized_Orders,Late_Realized_Orders,Late_Delivery_Rate
0,First Class,4497,4497,100.00
1,Second Class,5635,4513,80.09
2,Same Day,1548,751,48.51
3,Standard Class,17285,6872,39.76


In [163]:
#KPI:Orders and Delievery Status in Shipping Mode
con.execute("""
SELECT

    "Shipping Mode",

    "Delivery Status",

    COUNT(DISTINCT "Order ID") AS Orders

FROM supply_chain

WHERE "Order Status" IN ('CLOSED', 'COMPLETE')

GROUP BY

    "Shipping Mode",

    "Delivery Status"

ORDER BY

    "Shipping Mode",

    Orders DESC
""").df()

,Shipping Mode,Delivery Status,Orders
0,First Class,Late delivery,4497
1,Same Day,Shipping on time,797
2,Same Day,Late delivery,751
3,Second Class,Late delivery,4513
4,Second Class,Shipping on time,1122
5,Standard Class,Advance shipping,6951
6,Standard Class,Late delivery,6872
7,Standard Class,Shipping on time,3462


In [167]:

#KPI: Which markets have the highest late-delivery rate?


con.execute("""

SELECT

  "Market",

    COUNT(DISTINCT "Order ID") AS Total_Realized_Orders,

    COUNT(DISTINCT CASE

        WHEN "Delivery Status" = 'Late delivery'

        THEN "Order ID"

    END) AS Late_Realized_Orders,

    ROUND(

        COUNT(DISTINCT CASE

            WHEN "Delivery Status" = 'Late delivery'

            THEN "Order ID"

        END) * 100.0

        / COUNT(DISTINCT "Order ID"),

        2

    ) AS Late_Delivery_Rate

FROM supply_chain

WHERE "Order Status" IN ('CLOSED', 'COMPLETE')

GROUP BY "Market"

ORDER BY Late_Delivery_Rate DESC

""").df()






,Market,Total_Realized_Orders,Late_Realized_Orders,Late_Delivery_Rate
0,Pacific Asia,7673,4491,58.53
1,USCA,3813,2220,58.22
2,Europe,8232,4744,57.63
3,LATAM,7541,4223,56.00
4,Africa,1706,955,55.98


In [169]:
#KPI:How does late-delivery performance vary by shipping mode across different markets?

con.execute("""

SELECT

  "Market","Shipping mode",

    COUNT(DISTINCT "Order ID") AS Total_Realized_Orders,

    COUNT(DISTINCT CASE

        WHEN "Delivery Status" = 'Late delivery'

        THEN "Order ID"

    END) AS Late_Realized_Orders,

    ROUND(

        COUNT(DISTINCT CASE

            WHEN "Delivery Status" = 'Late delivery'

            THEN "Order ID"

        END) * 100.0

        / COUNT(DISTINCT "Order ID"),

        2

    ) AS Late_Delivery_Rate

FROM supply_chain

WHERE "Order Status" IN ('CLOSED', 'COMPLETE')

GROUP BY "Market","Shipping mode"

ORDER BY Late_Delivery_Rate DESC

""").df()






,Market,Shipping Mode,Total_Realized_Orders,Late_Realized_Orders,Late_Delivery_Rate
0,USCA,First Class,601,601,100.00
1,LATAM,First Class,1163,1163,100.00
2,Pacific Asia,First Class,1184,1184,100.00
3,Africa,First Class,238,238,100.00
4,Europe,First Class,1311,1311,100.00
5,Pacific Asia,Second Class,1530,1238,80.92
6,Europe,Second Class,1606,1295,80.64
7,USCA,Second Class,786,630,80.15
8,Africa,Second Class,317,250,78.86
9,LATAM,Second Class,1396,1100,78.80


## Profit Field Validation: Check Order Profit Per Order grain

In [189]:
#  DATA VALIDATION — Profit Field Grain

# Purpose:Determine how many distinct "Order Profit Per Order" values exist within each realized order.

# This helps us understand the granularity of the profit fielD before calculating category-level realized profit.

con.execute("""
select  Distinct_Profit_Values,
count(*) as number_of_orders
from
(Select "Order ID" as  Realized_Orders, Count(Distinct "Order Profit Per Order") as  Distinct_Profit_Values
from supply_chain
where "Order Status" = 'CLOSED' 
Or "Order Status" = 'COMPLETE'
Group by "Order ID") as k
Group by Distinct_Profit_Values


""").df()


,Distinct_Profit_Values,number_of_orders
0,2,5085
1,5,4966
2,4,5085
3,1,8875
4,3,4954


In [191]:
## Order-Level Product Grain Validation

con.execute("""

SELECT

    "Order ID",

    COUNT(*) AS Order_Rows,

    COUNT(DISTINCT "Product Name") AS Distinct_Products

FROM supply_chain

WHERE "Order Status" IN ('CLOSED', 'COMPLETE')

GROUP BY "Order ID"

ORDER BY Order_Rows DESC

LIMIT 20

""").df()

,Order Id,Order_Rows,Distinct_Products
0,21991,5,3
1,19831,5,3
2,21776,5,4
3,19362,5,5
4,19342,5,5
5,18879,5,5
6,18867,5,3
7,19237,5,4
8,20794,5,4
9,19319,5,4


## Category Proit and sales Analysis

In [200]:

#KPI Category Proit and sales Contribution
con.execute("""
SELECT
    "Category Name" AS Product_Category,

    ROUND(
        SUM(Sales) / 1000000.0,
        2
    ) AS Realized_Sales_M,

    ROUND(
        SUM("Order Profit Per Order")
        / NULLIF(SUM(Sales), 0) * 100,
        2
    ) AS Profit_Margin_Pct,

    ROUND(
        COUNT(DISTINCT CASE
            WHEN "Delivery Status" = 'Late delivery'
            THEN "Order ID"
        END) * 100.0
        / NULLIF(COUNT(DISTINCT "Order ID"), 0),
        2
    ) AS Late_Delivery_Rate

FROM supply_chain

WHERE "Order Status" IN ('CLOSED', 'COMPLETE')

GROUP BY "Category Name"

ORDER BY Realized_Sales_M DESC

LIMIT 5
""").df()

,Product_Category,Realized_Sales_M,Profit_Margin_Pct,Late_Delivery_Rate
0,Fishing,3.02,11.06,56.83
1,Cleats,1.93,11.67,58.15
2,Camping & Hiking,1.80,10.52,56.37
3,Cardio Equipment,1.64,10.99,57.35
4,Women's Apparel,1.39,11.53,56.39


In [212]:
# Extracting Category Proit and sales Contribution into a DataFrame

kpi_Category_Proit_and_sales = con.execute("""

SELECT

    product_cat,

    ROUND(Realized_Sales / 1000000, 2) AS Realized_Sales_M,

    ROUND(
        (Realized_Sales / Total_Sales) * 100,
        2
    ) AS Sales_Contribution_Pct,

    ROUND(Realized_Profit / 1000000, 2) AS Realized_Profit_M,

    ROUND(
        (Realized_Profit / Realized_Sales) * 100,
        2
    ) AS Profit_Margin_Pct,

    Total_Realized_Orders,

    Late_Realized_Orders,

    Late_Delivery_Rate,

    Average_Shipping_Delay

FROM (

    SELECT

        "Category Name" AS product_cat,

        SUM(Sales) AS Realized_Sales,

        SUM("Order Profit Per Order") AS Realized_Profit,

        SUM(SUM(Sales)) OVER () AS Total_Sales,

        COUNT(DISTINCT "Order ID") AS Total_Realized_Orders,

        COUNT(
            DISTINCT CASE
                WHEN "Delivery Status" = 'Late delivery'
                THEN "Order ID"
            END
        ) AS Late_Realized_Orders,

        ROUND(
            COUNT(
                DISTINCT CASE
                    WHEN "Delivery Status" = 'Late delivery'
                    THEN "Order ID"
                END
            ) * 100.0
            / COUNT(DISTINCT "Order ID"),
            2
        ) AS Late_Delivery_Rate,

        ROUND(
            AVG(
                "Days for shipping (real)"
                - "Days for shipment (scheduled)"
            ),
            2
        ) AS Average_Shipping_Delay

    FROM supply_chain

    WHERE "Order Status" IN ('CLOSED', 'COMPLETE')

    GROUP BY "Category Name"

) AS k

ORDER BY Late_Delivery_Rate DESC

""").df()

In [214]:
kpi_Category_Proit_and_sales.to_excel("KPI_Commercial_Impact_Late_Delivery.xlsx", index=False)

In [216]:
import os

os.path.exists("KPI_Commercial_Impact_Late_Delivery.xlsx")

True

In [218]:
from IPython.display import FileLink

FileLink("KPI_Commercial_Impact_Late_Delivery.xlsx")

/Users/shivanimaithani/KPI_Commercial_Impact_Late_Delivery.xlsx

## Region Performance

In [222]:
# KPI Market Delivery Performance & Commercial Exposure
# Objective:Identify markets with high late-delivery rates and determine whether those markets also represent significant commercial exposure.

con.execute("""

SELECT
    Market,

    -- Delivery performance
    COUNT(*) AS Total_Realized_Orders,

    SUM(
        CASE
            WHEN "Days for shipping (real)"
                 > "Days for shipment (scheduled)"
            THEN 1
            ELSE 0
        END
    ) AS Late_Orders,

    ROUND(
        SUM(
            CASE
                WHEN "Days for shipping (real)"
                     > "Days for shipment (scheduled)"
                THEN 1
                ELSE 0
            END
        ) * 100.0 / COUNT(*),
        2
    ) AS Late_Delivery_Rate,

    -- Commercial exposure
    ROUND(SUM(Sales) / 1000000, 2) AS Realized_Sales_M,

    ROUND(
        SUM(Sales) * 100.0 /
        SUM(SUM(Sales)) OVER (),
        2
    ) AS Sales_Contribution_Pct,

    ROUND(
        SUM("Order Profit Per Order") / 1000000,
        2
    ) AS Realized_Profit_M,

    ROUND(
        SUM("Order Profit Per Order") * 100.0 /
        NULLIF(SUM(Sales), 0),
        2
    ) AS Profit_Margin_Pct

FROM supply_chain

WHERE "Order Status" IN ('CLOSED', 'COMPLETE')

GROUP BY Market

ORDER BY Late_Delivery_Rate DESC

""").df()

,Market,Total_Realized_Orders,Late_Orders,Late_Delivery_Rate,Realized_Sales_M,Sales_Contribution_Pct,Realized_Profit_M,Profit_Margin_Pct
0,USCA,11575,6739.0,58.22,2.26,14.03,0.25,10.95
1,Pacific Asia,17822,10353.0,58.09,3.58,22.18,0.39,10.84
2,Europe,22093,12804.0,57.96,4.82,29.93,0.55,11.42
3,Africa,5190,2915.0,56.17,1.01,6.29,0.11,11.08
4,LATAM,22427,12497.0,55.72,4.44,27.57,0.48,10.84


## Data coverage validation

In [236]:
# Data coverage validation

coverage_df = con.execute("""

SELECT

    EXTRACT(YEAR FROM Order_Date) AS Year,

    MIN(Order_Date) AS First_Date,

    MAX(Order_Date) AS Last_Date,

    COUNT(DISTINCT DATE_TRUNC('month', Order_Date)) AS Months_Present,

    COUNT(*) AS Rows

FROM (

    SELECT

        STRPTIME(

            "order date (DateOrders)",

            '%m/%d/%Y %H:%M'

        ) AS Order_Date

    FROM supply_chain

)

GROUP BY EXTRACT(YEAR FROM Order_Date)

ORDER BY Year

""").df()

coverage_df

print("""

Data Coverage Assessment:

- 2015, 2016 and 2017 have 12 months of observations and are treated as full-year coverage.

- 2018 contains observations for January only and is treated as partial-year coverage.

- 2018 is therefore excluded from full-year YoY comparisons.

""")



Data Coverage Assessment:

- 2015, 2016 and 2017 have 12 months of observations and are treated as full-year coverage.

- 2018 contains observations for January only and is treated as partial-year coverage.

- 2018 is therefore excluded from full-year YoY comparisons.




## Monthly Delivery Performance Analysis

In [259]:
kpi_Monthly_Performance_df = con.execute("""

WITH order_level AS (

    SELECT

        "Order Id",

        DATE_TRUNC('month', STRPTIME("order date (DateOrders)", '%m/%d/%Y %H:%M')) AS Order_Month,

        MAX(CASE WHEN "Order Status" IN ('CLOSED', 'COMPLETE') THEN 1 ELSE 0 END) AS Is_Realised,

        MAX(CASE WHEN "Delivery Status" = 'Late delivery' THEN 1 ELSE 0 END) AS Is_Late,

        SUM(Sales) AS Order_Sales,

        SUM("Order Profit Per Order") AS Order_Profit

    FROM supply_chain

    GROUP BY

        "Order Id",

        DATE_TRUNC('month', STRPTIME("order date (DateOrders)", '%m/%d/%Y %H:%M'))

),

monthly_performance AS (

    SELECT

        Order_Month,

        COUNT(*) AS Total_Orders,

        SUM(Is_Realised) AS Realised_Orders,

        SUM(CASE WHEN Is_Realised = 1 AND Is_Late = 1 THEN 1 ELSE 0 END) AS Late_Orders,

        SUM(CASE WHEN Is_Realised = 1 AND Is_Late = 0 THEN 1 ELSE 0 END) AS On_Time_Orders,

        SUM(CASE WHEN Is_Realised = 1 THEN Order_Sales ELSE 0 END) AS Realised_Sales,

        SUM(CASE WHEN Is_Realised = 1 THEN Order_Profit ELSE 0 END) AS Realised_Profit

    FROM order_level

    GROUP BY Order_Month

),

monthly_kpis AS (

    SELECT

        Order_Month,

        Total_Orders,

        Realised_Orders,

        ROUND(Realised_Orders * 100.0 / NULLIF(Total_Orders, 0), 2) AS Realisation_Rate,

        Late_Orders,

        ROUND(Late_Orders * 100.0 / NULLIF(Realised_Orders, 0), 2) AS Late_Delivery_Rate,

        On_Time_Orders,

        ROUND(On_Time_Orders * 100.0 / NULLIF(Realised_Orders, 0), 2) AS On_Time_Delivery_Rate,

        ROUND(Realised_Sales / 1000000.0, 2) AS Realised_Sales_M,

        ROUND(Realised_Profit / 1000000.0, 2) AS Realised_Profit_M,

        ROUND(Realised_Profit * 100.0 / NULLIF(Realised_Sales, 0), 2) AS Profit_Margin

    FROM monthly_performance

),

monthly_with_previous AS (

    SELECT

        *,

        LAG(Late_Delivery_Rate) OVER (ORDER BY Order_Month) AS Previous_Month_Late_Delivery_Rate,

        LAG(Realisation_Rate) OVER (ORDER BY Order_Month) AS Previous_Month_Realisation_Rate

    FROM monthly_kpis

)

SELECT

    Order_Month,

    Total_Orders,

    Realised_Orders,

    Realisation_Rate,

    Late_Orders,

    Late_Delivery_Rate,

    On_Time_Orders,

    On_Time_Delivery_Rate,

    Realised_Sales_M,

    Realised_Profit_M,

    Profit_Margin,

    ROUND(Late_Delivery_Rate - Previous_Month_Late_Delivery_Rate, 2) AS MoM_Late_Delivery_Change_pp,

    ROUND(Realisation_Rate - Previous_Month_Realisation_Rate, 2) AS MoM_Realisation_Rate_Change_pp

FROM monthly_with_previous

WHERE Order_Month < '2018-01-01'

ORDER BY Order_Month

""").df()

kpi_Monthly_Performance_df

,Order_Month,Total_Orders,Realised_Orders,Realisation_Rate,Late_Orders,Late_Delivery_Rate,On_Time_Orders,On_Time_Delivery_Rate,Realised_Sales_M,Realised_Profit_M,Profit_Margin,MoM_Late_Delivery_Change_pp,MoM_Realisation_Rate_Change_pp
0,2015-01-01,1787,779.0,43.59,417.0,53.53,362.0,46.47,0.45,0.05,10.97,NaN,NaN
1,2015-02-01,1585,695.0,43.85,399.0,57.41,296.0,42.59,0.41,0.04,10.10,3.88,0.26
2,2015-03-01,1781,778.0,43.68,435.0,55.91,343.0,44.09,0.45,0.05,10.71,-1.50,-0.17
3,2015-04-01,1710,761.0,44.50,425.0,55.85,336.0,44.15,0.44,0.05,11.09,-0.06,0.82
4,2015-05-01,1776,783.0,44.09,442.0,56.45,341.0,43.55,0.47,0.05,10.04,0.60,-0.41
5,2015-06-01,1725,769.0,44.58,438.0,56.96,331.0,43.04,0.45,0.05,11.59,0.51,0.49
6,2015-07-01,1763,788.0,44.70,448.0,56.85,340.0,43.15,0.46,0.05,10.57,-0.11,0.12
7,2015-08-01,1762,786.0,44.61,461.0,58.65,325.0,41.35,0.47,0.06,11.80,1.80,-0.09
8,2015-09-01,1706,769.0,45.08,447.0,58.13,322.0,41.87,0.46,0.06,12.79,-0.52,0.47
9,2015-10-01,1775,770.0,43.38,435.0,56.49,335.0,43.51,0.45,0.04,9.54,-1.64,-1.70


In [261]:
kpi_Monthly_Performance_df.to_excel(

    "KPI_Monthly_Delivery_Performance.xlsx",

    index=False

)

In [263]:
from IPython.display import FileLink

FileLink("KPI_Monthly_Delivery_Performance.xlsx")

/Users/shivanimaithani/KPI_Monthly_Delivery_Performance.xlsx

## Customer Segment Delivery Performance & Commercial Exposure

In [254]:
# KPI  Customer Segment Delivery Performance & Commercial Exposure

#

# Objective:

# Identify customer segments with higher late-delivery rates

# and assess whether those segments represent significant

# commercial exposure.

#

# Key measures:

# - Realised Orders

# - Late Realised Orders

# - Late Delivery Rate

# - Realised Sales

# - Sales Contribution

# - Realised Profit

# - Profit Margin

kpi_Customer_Segment_Analysis_df = con.execute("""

WITH order_level AS (

    SELECT

        "Order Id",

        "Customer Segment" AS Customer_Segment,

        MAX(

            CASE

                WHEN "Delivery Status" = 'Late delivery'

                THEN 1

                ELSE 0

            END

        ) AS Is_Late,

        SUM(Sales) AS Order_Sales,

        SUM("Order Profit Per Order") AS Order_Profit

    FROM supply_chain

    WHERE "Order Status" IN ('CLOSED', 'COMPLETE')

    GROUP BY

        "Order Id",

        "Customer Segment"

),

segment_performance AS (

    SELECT

        Customer_Segment,

        COUNT(*) AS Total_Realized_Orders,

        SUM(Is_Late) AS Late_Realized_Orders,

        SUM(Order_Sales) AS Realized_Sales,

        SUM(Order_Profit) AS Realized_Profit

    FROM order_level

    GROUP BY

        Customer_Segment

)

SELECT

    Customer_Segment,

    Total_Realized_Orders,

    Late_Realized_Orders,

    ROUND(

        Late_Realized_Orders * 100.0

        / NULLIF(Total_Realized_Orders, 0),

        2

    ) AS Late_Delivery_Rate,

    ROUND(

        Realized_Sales / 1000000.0,

        2

    ) AS Realized_Sales_M,

    ROUND(

        Realized_Sales * 100.0

        / SUM(Realized_Sales) OVER (),

        2

    ) AS Sales_Contribution_Pct,

    ROUND(

        Realized_Profit / 1000000.0,

        2

    ) AS Realized_Profit_M,

    ROUND(

        Realized_Profit * 100.0

        / NULLIF(Realized_Sales, 0),

        2

    ) AS Profit_Margin_Pct

FROM segment_performance

ORDER BY

    Late_Delivery_Rate DESC

""").df()

kpi_Customer_Segment_Analysis_df

,Customer_Segment,Total_Realized_Orders,Late_Realized_Orders,Late_Delivery_Rate,Realized_Sales_M,Sales_Contribution_Pct,Realized_Profit_M,Profit_Margin_Pct
0,Home Office,5184,3008.0,58.02,2.86,17.72,0.30,10.56
1,Consumer,15000,8616.0,57.44,8.34,51.74,0.93,11.17
2,Corporate,8781,5009.0,57.04,4.92,30.54,0.55,11.10
